# Artificial Membrane practical
## Estimate the permeability ratio $P_K / P_{Cl}$

Enter the potential difference (mV) you measured for each ECF [KCl] you tested, then click **Compute**.

- Leave a field blank for any concentration your group didn't measure.
- Enter the value exactly as measured, **including its sign**. Cation-selective membranes typically give negative voltages; anion-selective membranes give positive voltages — the fit works either way and will report which ion the membrane is more permeable to.

In [ ]:
try:
    import piplite
    await piplite.install(["numpy", "matplotlib", "ipywidgets"])
except ImportError:
    pass  # not running under Pyodide/JupyterLite; assume these are already installed

import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display

from artmem import ECF_CONCENTRATIONS, ICF_CONCENTRATION, fit_permeability_ratio, plot_KCl_activity

%matplotlib inline

In [ ]:
conc_inputs = {
    c: widgets.Text(
        value="",
        placeholder="mV",
        description=f"{c} mM:",
        style={"description_width": "70px"},
        layout=widgets.Layout(width="200px"),
    )
    for c in ECF_CONCENTRATIONS
}
compute_button = widgets.Button(description="Compute P_K / P_Cl", button_style="primary")
output = widgets.Output()

form = widgets.VBox(
    [widgets.HTML(f"<b>ECF [KCl] (ICF fixed at {ICF_CONCENTRATION} mM) \u2192 measured V (mV):</b>")]
    + list(conc_inputs.values())
    + [compute_button, output]
)


def on_compute_clicked(_):
    output.clear_output()
    concs, volts = [], []
    for c, box in conc_inputs.items():
        text = box.value.strip()
        if text == "":
            continue
        try:
            volts.append(float(text))
            concs.append(c)
        except ValueError:
            with output:
                print(f"Could not parse '{text}' for {c} mM \u2014 please enter a number.")
            return
    with output:
        if len(concs) < 2:
            print("Enter at least two measured concentrations.")
            return
        result = fit_permeability_ratio(concs, volts)
        plt.show()
        print(f"Estimated P_K / P_Cl = {result['r_est']:.3f}")


compute_button.on_click(on_compute_clicked)
display(form)

---
### Optional: KCl activity vs. concentration

The Nernst-Planck fit uses chemical *activity*, not concentration — real ions in solution don't behave quite ideally. This plot shows how much the two diverge over the range used in this practical.

In [ ]:
plot_KCl_activity(ECF_CONCENTRATIONS)
plt.show()

---
### Optional: simulate demo data (for testing/demonstration only)

This cell generates noisy simulated measurements for a chosen "true" permeability ratio, so you can see how the fit behaves before working with real data. It is not needed for the actual practical.

In [ ]:
from artmem import fit_permeability_ratio, get_activity, cRTF

r_true = 10  # try e.g. 0.1 (anion-selective) or 10 (cation-selective)
concs_demo = ECF_CONCENTRATIONS
a_ecf_demo = np.array([get_activity(c) for c in concs_demo])
a_icf_demo = get_activity(ICF_CONCENTRATION)

V_demo = 1e3 * (r_true - 1) / (r_true + 1) * cRTF * np.log10(a_ecf_demo / a_icf_demo)
V_demo = V_demo + 5 * np.random.randn(len(V_demo))  # add measurement noise

fit_permeability_ratio(concs_demo, V_demo, r_true=r_true)
plt.show()